# Rouge 1,2,L benchmarks

This notebook calculates the rouge benchmarks for the fine tuned ml model.
This benchmark is based on the reference paper. Therefore also scispacy is used to calculate a more accurate benchmark of titles.

**Note:** Use python3.11 to execute this notebook. Otherwise `sciSpacy` won't work.

In [ ]:
%pip install spacy rouge-score pandas torch transformers peft 
# Install the en_core_sci_sm model for scispacy
%pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz (14.8 MB)
  Preparing metadata (setup.py) ... done

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [1]:
import spacy
from rouge_score import rouge_scorer
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

# SciSpaCy model
nlp = spacy.load("en_core_sci_sm")

test_df = pd.read_csv("test_LREC_COLING.csv")

test_df

,Abstract,Title
0,Multimodal machine translation (MMT) is a chal...,3AM: An Ambiguity-Aware Multi-Modal Machine Tr...
1,Background: Transformer-based language models ...,A Benchmark Evaluation of Clinical Named Entit...
2,This paper introduces a novel benchmark that h...,A Benchmark for Recipe Understanding in Artifi...
3,Natural Language Processing (NLP) models tend ...,ABLE: Agency-BeLiefs Embedding to Address Ster...
4,"This paper introduces a new task, abstractive ...",Abstractive Multi-Video Captioning: Benchmark ...
...,...,...
995,Artificial intelligence (AI)-aided disease pre...,MKeCL: Medical Knowledge-Enhanced Contrastive ...
996,The intelligent chatbot takes dialogue sentime...,MLDSP-MA: Multidimensional Attention for Multi...
997,Audio Description (AD) aims to generate narrat...,MMAD:Multi-modal Movie Audio Description
998,Given the long textual product information and...,MMAPS: End-to-End Multi-Grained Multi-Modal At...


In [ ]:
BASE_MODEL = "HuggingFaceTB/SmolLM-135M"
# Change this to load different models
ADAPTER_PATH = "../../runs/lora-own"

In [5]:
def load():
    """
    Load the model, tokenizer and determine device (mps or cpu)
    """
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)

    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

    device = "mps" if torch.mps.is_available() else "cpu"
    model = model.to(device)
    model.eval()

    return model, tokenizer, device

# Initialize LM
model, tokenizer, device = load()

In [ ]:
def predict(text):
    """
    Takes in the abstract and generates a title for it.
    Only returns the title! (Abstract is cutted away)
    Max number of tokens is set to 30. -> This diverges from the paper, however,
    the new trainingset includes much longer titles as well, therefore it is only
    fair to let the model "finish" generating its title.
    """
    prompt = f"{text}\n\nTitle: "

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=30,
            top_p=0.9,
            temperature=0.7,
            repetition_penalty=1.2,

            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = out[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return decoded.strip()


In [7]:
def extract_entities(text):
    """
    Identifies named entities from the given text.
    What an entity is or is not, is defined by `en_core_sci_sm`.
    Returns named entities as a list.
    """
    doc = nlp(text)
    return [ent.text.lower() for ent in doc.ents]


def entity_match(e1, e2):
    """
    Returns true if both entities partially match.
    E.g. "Neural Networks" and "Neural" => true.
    """
    tokens1 = set(e1.split())
    tokens2 = set(e2.split())
    return len(tokens1 & tokens2) > 0


def count_matches_list(x, y):
    count = 0
    for ex in x:
        for ey in y:
            if entity_match(ex, ey):
                count += 1
                break
    return count


def count_matches_set(x, y):
    x_set = set(x)
    y_set = set(y)

    count = 0
    for ex in x_set:
        for ey in y_set:
            if entity_match(ex, ey):
                count += 1
                break
    return count


# Metrics (exact paper version)

def compute_entity_metrics(Eh, Et, Es):
    """
    used to check FACTUAL CONSISTENCY

    h = hypothesis (or predicted title)
    t = truth (or actual title)
    s = source (or abstract/text)

    u = unique (multiple occurrences of a named entity count only once)
    nu = non unique (multiple occurrences of a named entity count multiple times)

    precision = Of all the items the model labeled as positive, how many were actually positive?
    recall = Of all the actual positives, how many did the model correctly identify?
    f1 = balance out of precision and recall
    """
    def safe_div(a, b):
        return a / b if b > 0 else 0.0

    # UNIQUE (U)
    Nh_u = len(set(Eh))
    Nt_u = len(set(Et))

    h_s_u = count_matches_set(Eh, Es)
    h_t_u = count_matches_set(Eh, Et)

    prec_s_u = safe_div(h_s_u, Nh_u)
    prec_t_u = safe_div(h_t_u, Nh_u)
    rec_t_u = safe_div(h_t_u, Nt_u)
    f1_t_u = safe_div(2 * prec_t_u * rec_t_u, prec_t_u + rec_t_u)

    # NON-UNIQUE (NU)
    Nh = len(Eh)
    Nt = len(Et)

    h_s = count_matches_list(Eh, Es)
    h_t = count_matches_list(Eh, Et)

    prec_s_nu = safe_div(h_s, Nh)
    prec_t_nu = safe_div(h_t, Nh)
    rec_t_nu = safe_div(h_t, Nt)
    f1_t_nu = safe_div(2 * prec_t_nu * rec_t_nu, prec_t_nu + rec_t_nu)

    return {
        "prec_s_u": prec_s_u,
        "prec_t_u": prec_t_u,
        "rec_t_u": rec_t_u,
        "f1_t_u": f1_t_u,
        "prec_s_nu": prec_s_nu,
        "prec_t_nu": prec_t_nu,
        "rec_t_nu": rec_t_nu,
        "f1_t_nu": f1_t_nu,
    }

In [9]:
rouge1 = 0.0
rouge2 = 0.0
rougeL = 0.0

entity_scores = {
    k: 0.0 for k in [
        "prec_s_u", "prec_t_u", "rec_t_u", "f1_t_u",
        "prec_s_nu", "prec_t_nu", "rec_t_nu", "f1_t_nu"
    ]
}

for row in test_df.itertuples():
    prediction = predict(row.Abstract).lower()
    ground_truth = row.Title.lower().strip()

    # Calculate rouge scores between generated title and actual title
    scores = scorer.score(ground_truth, prediction)

    s1 = scores['rouge1'].fmeasure
    s2 = scores['rouge2'].fmeasure
    sl = scores['rougeL'].fmeasure

    rouge1 += s1
    rouge2 += s2
    rougeL += sl

    # ENTITY METRICS
    Eh = extract_entities(prediction)    # hypothesis
    Et = extract_entities(ground_truth)  # ground truth
    Es = extract_entities(row.Abstract)  # source

    metrics = compute_entity_metrics(Eh, Et, Es)

    for k in entity_scores:
        entity_scores[k] += metrics[k]

    print(s1, s2, sl)
    # Debug title difference if prediction does absolutely not match.
    if s1 + s2 + sl == 0:
        print("Original:", ground_truth, "\nGenerated:", prediction)


0.4545454545454546 0.3 0.4545454545454546
0.75 0.5454545454545455 0.6666666666666666
0.2 0.1111111111111111 0.2
0.42857142857142855 0.15384615384615383 0.28571428571428575
0.6666666666666665 0.6250000000000001 0.6666666666666665
0.09523809523809525 0.0 0.09523809523809525
0.07692307692307691 0.0 0.07692307692307691
0.3157894736842105 0.11764705882352942 0.3157894736842105
0.08695652173913043 0.0 0.08695652173913043
0.5714285714285715 0.5263157894736842 0.5714285714285715
0.41666666666666663 0.09090909090909093 0.16666666666666666
0.3 0.0 0.3
0.27272727272727276 0.1 0.27272727272727276
0.2105263157894737 0.11764705882352941 0.2105263157894737
0.39999999999999997 0.33333333333333326 0.39999999999999997
0.28571428571428575 0.0 0.21428571428571427
0.5714285714285715 0.23076923076923075 0.42857142857142855
0.07407407407407407 0.0 0.07407407407407407
0.3333333333333333 0.09090909090909093 0.25
0.4615384615384615 0.1818181818181818 0.30769230769230765
0.16666666666666669 0.0 0.166666666666666

In [10]:
n = len(test_df)

print("\nFinal ROUGE Results:")
print("F1 Rouge 1:", rouge1 / n)
print("F1 Rouge 2:", rouge2 / n)
print("F1 Rouge L:", rougeL / n)

print("\nEntity-level factual consistency:")

for k, v in entity_scores.items():
    print(f"{k}: {v / n:.4f}")


Final ROUGE Results:
F1 Rouge 1: 0.2085480406074193
F1 Rouge 2: 0.07353052721940127
F1 Rouge L: 0.17109916732524624

Entity-level factual consistency:
prec_s_u: 0.4796
prec_t_u: 0.2196
rec_t_u: 0.2206
f1_t_u: 0.2078
prec_s_nu: 0.4799
prec_t_nu: 0.2199
rec_t_nu: 0.2221
f1_t_nu: 0.2085
